# 06 — 최종 모델 비교 및 핵심 질문 답변

SMAPE(예측 정확도)와 TotalCost(운영 비용) 두 축으로 세 모델을 종합 비교하고,  
프로젝트 핵심 질문에 데이터 기반으로 답합니다.

> **핵심 질문: "예측 정확도가 높을수록 실제 운영 비용도 반드시 감소하는가?"**

---
| 셀 | 내용 |
|---|---|
| 3 | 전체 결과 로드 및 통합 |
| 4 | SMAPE × TotalCost 2×2 매트릭스 |
| 5 | 레이더 차트 (다차원 모델 비교) |
| 6 | 카테고리별 최적 모델 지도 |
| 7 | 비용 절감 효과 분석 |
| 8 | 핵심 질문 — 데이터 기반 답변 |
| 9 | 프로젝트 최종 요약 대시보드 |

## 1. 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi'        : 130,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.25,
    'font.size'         : 9,
})

MODEL_COLORS = {
    'ARIMA'   : '#378ADD',
    'XGBoost' : '#1D9E75',
    'LSTM'    : '#D85A30',
    'Perfect' : '#7F77DD',
    'Naive'   : '#AAAAAA',
}
MODELS = ['ARIMA', 'XGBoost', 'LSTM']

print("라이브러리 로드 완료")

## 2. 이전 단계 데이터 전체 재생성

> 이전 노트북과 이어서 실행하는 경우 건너뛰어도 됩니다.

In [ ]:
# ── 공통 설정 ────────────────────────────────────────
PATH           = '../data/'
ANALYSIS_START = '2017-01-01'
ANALYSIS_END   = '2018-08-31'
TRAIN_END      = '2018-06-30'
TEST_START     = '2018-07-01'
TOP_N          = 10
WINDOW_SIZE    = 14
H_SCALE        = 0.1
S_RATIO        = 3.0
STRATEGY       = 'newsvendor'
FEATURE_COLS   = [
    'lag_1','lag_7','lag_14',
    'rolling_mean_7','rolling_std_7','rolling_mean_14',
    'weekday','is_weekend','month','day_of_year'
]

# ── 저장된 결과 로드 ─────────────────────────────────
pred_df  = pd.read_csv(PATH + 'predictions.csv',  parse_dates=['date'])
smape_df = pd.read_csv(PATH + 'smape_summary.csv')
CATEGORIES = pred_df['category'].unique().tolist()

# ── 배송비 기반 h·s 재계산 ──────────────────────────
order_items = pd.read_csv(PATH + 'olist_order_items_dataset.csv')
products    = pd.read_csv(PATH + 'olist_products_dataset.csv')
merged_cost = (
    order_items[['order_id','product_id','freight_value']]
    .merge(products[['product_id','product_category_name']], on='product_id')
    .dropna(subset=['product_category_name'])
)
freight_by_cat = merged_cost.groupby('product_category_name')['freight_value'].mean()

cost_params = {}
for cat in CATEGORIES:
    avg_f = freight_by_cat.get(cat, freight_by_cat.mean())
    h     = avg_f * H_SCALE
    cost_params[cat] = {'h': h, 's': h * S_RATIO}

# ── 비용 함수 ────────────────────────────────────────
from scipy.stats import norm

def order_quantity(pred, strategy, h, s, rolling_std=None):
    pred = np.maximum(0, np.array(pred, dtype=float))
    if strategy == 'exact':
        return pred
    q   = s / (s + h)
    z   = norm.ppf(q)
    std = rolling_std if rolling_std is not None else np.ones_like(pred)
    return np.maximum(0, pred + z * std)

def simulate_cost(actual, order, h, s):
    actual, order = np.array(actual, dtype=float), np.array(order, dtype=float)
    holding  = np.maximum(order - actual, 0) * h
    stockout = np.maximum(actual - order, 0) * s
    total    = holding + stockout
    return {
        'holding_daily' : holding,  'stockout_daily': stockout, 'total_daily': total,
        'holding_sum'   : holding.sum(), 'stockout_sum': stockout.sum(), 'total_sum': total.sum(),
    }

# ── 비용 시뮬레이션 재실행 ───────────────────────────
pred_col_map = {'ARIMA':'pred_arima','XGBoost':'pred_xgb','LSTM':'pred_lstm'}
cost_results = {}
cost_daily_rows = []

for cat in CATEGORIES:
    sub = pred_df[pred_df['category']==cat].sort_values('date')
    sub = sub.dropna(subset=list(pred_col_map.values()))
    actual      = sub['actual'].values
    dates       = sub['date'].values
    h, s        = cost_params[cat]['h'], cost_params[cat]['s']
    rolling_std = pd.Series(actual).rolling(7, min_periods=2).std().fillna(1).values
    cost_results[cat] = {}

    for model, col in pred_col_map.items():
        order  = order_quantity(sub[col].values, STRATEGY, h, s, rolling_std)
        result = simulate_cost(actual, order, h, s)
        cost_results[cat][model] = result
        for t, dt in enumerate(dates):
            cost_daily_rows.append({
                'date':dt,'category':cat,'model':model,
                'actual':actual[t],'order':order[t],
                'holding':result['holding_daily'][t],
                'stockout':result['stockout_daily'][t],
                'total':result['total_daily'][t],
            })

    # Perfect / Naive
    cost_results[cat]['Perfect'] = simulate_cost(
        actual, order_quantity(actual, STRATEGY, h, s, rolling_std), h, s)
    naive_pred = np.concatenate([[actual[0]], actual[:-1]])
    cost_results[cat]['Naive'] = simulate_cost(
        actual, order_quantity(naive_pred, STRATEGY, h, s, rolling_std), h, s)

cost_daily_df = pd.DataFrame(cost_daily_rows)

print("전체 데이터 로드 및 비용 재계산 완료")
print(f"카테고리 {len(CATEGORIES)}개 / 예측 기간 {TEST_START} ~ {ANALYSIS_END}")

## 3. 통합 결과 테이블 생성

In [ ]:
def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    return float(np.mean(np.where(denom!=0, np.abs(y_true-y_pred)/denom*100, 0)))

# 카테고리 × 모델 통합 결과
summary_rows = []
for cat in CATEGORIES:
    sub = pred_df[pred_df['category']==cat].dropna(subset=list(pred_col_map.values()))
    for model, col in pred_col_map.items():
        r = cost_results[cat][model]
        summary_rows.append({
            'category'    : cat,
            'model'       : model,
            'smape'       : smape(sub['actual'], sub[col]),
            'total_cost'  : r['total_sum'],
            'holding_cost': r['holding_sum'],
            'stockout_cost': r['stockout_sum'],
            'perfect_cost': cost_results[cat]['Perfect']['total_sum'],
            'naive_cost'  : cost_results[cat]['Naive']['total_sum'],
        })

summary = pd.DataFrame(summary_rows)

# 카테고리별 최적 모델 (SMAPE / 비용 각각)
best_smape_model = summary.loc[summary.groupby('category')['smape'].idxmin(),
                                ['category','model']].rename(columns={'model':'best_smape_model'})
best_cost_model  = summary.loc[summary.groupby('category')['total_cost'].idxmin(),
                                ['category','model']].rename(columns={'model':'best_cost_model'})

cat_summary = best_smape_model.merge(best_cost_model, on='category')
cat_summary['reversed'] = cat_summary['best_smape_model'] != cat_summary['best_cost_model']

print("통합 결과 테이블 생성 완료")
print(f"전체 행 수: {len(summary)}  ({len(CATEGORIES)} 카테고리 × 3 모델)")
summary.groupby('model')[['smape','total_cost']].mean().round(2)

## 4. SMAPE × TotalCost 2×2 매트릭스

전체 평균을 기준으로 4사분면으로 나눕니다.

| 사분면 | 의미 |
|---|---|
| 좌하 (Low SMAPE, Low Cost) | **이상적** — 정확하고 비용도 낮음 |
| 좌상 (Low SMAPE, High Cost) | **역전** — 정확하지만 비용이 높음 |
| 우하 (High SMAPE, Low Cost) | **역전** — 부정확하지만 비용은 낮음 |
| 우상 (High SMAPE, High Cost) | **최악** — 부정확하고 비용도 높음 |

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

smape_mid = summary['smape'].mean()
cost_mid  = summary['total_cost'].mean()

# 사분면 배경색
xlim = (summary['smape'].min()*0.85,   summary['smape'].max()*1.1)
ylim = (summary['total_cost'].min()*0.8, summary['total_cost'].max()*1.1)

ax.fill_betweenx([ylim[0], cost_mid],  xlim[0], smape_mid,  alpha=0.06, color='#1D9E75')  # 좌하: 이상
ax.fill_betweenx([cost_mid, ylim[1]],  xlim[0], smape_mid,  alpha=0.06, color='#D85A30')  # 좌상: 역전
ax.fill_betweenx([ylim[0], cost_mid],  smape_mid, xlim[1],  alpha=0.06, color='#D85A30')  # 우하: 역전
ax.fill_betweenx([cost_mid, ylim[1]],  smape_mid, xlim[1],  alpha=0.06, color='#888888')  # 우상: 최악

# 기준선
ax.axvline(smape_mid, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
ax.axhline(cost_mid,  color='gray', linewidth=0.8, linestyle='--', alpha=0.6)

# 사분면 라벨
quad_kw = dict(fontsize=8, alpha=0.5, fontweight='bold', ha='center')
ax.text((xlim[0]+smape_mid)/2, ylim[0]*1.02, '◀ 이상 (정확 + 저비용)',    color='#1D9E75', **quad_kw)
ax.text((smape_mid+xlim[1])/2, ylim[0]*1.02, '최악 (부정확 + 고비용) ▶',  color='#888888', **quad_kw)
ax.text((xlim[0]+smape_mid)/2, ylim[1]*0.97, '역전 (정확하나 고비용)',     color='#D85A30', **quad_kw)
ax.text((smape_mid+xlim[1])/2, ylim[1]*0.97, '역전 (부정확하나 저비용)',   color='#D85A30', **quad_kw)

# 산점도
for model in MODELS:
    sub = summary[summary['model'] == model]
    ax.scatter(sub['smape'], sub['total_cost'],
               color=MODEL_COLORS[model], s=90, alpha=0.9,
               edgecolors='white', linewidths=0.5,
               label=model, zorder=4)
    for _, row in sub.iterrows():
        ax.annotate(row['category'][:12],
                    (row['smape'], row['total_cost']),
                    fontsize=6, alpha=0.65,
                    xytext=(4, 4), textcoords='offset points')

# 모델별 평균 마커 (크게)
for model in MODELS:
    sub  = summary[summary['model'] == model]
    mx, my = sub['smape'].mean(), sub['total_cost'].mean()
    ax.scatter(mx, my, color=MODEL_COLORS[model],
               s=250, marker='D', edgecolors='black',
               linewidths=1.2, zorder=6)
    ax.annotate(f'{model}\n평균',
                (mx, my), fontsize=8, fontweight='bold',
                xytext=(8, -14), textcoords='offset points',
                color=MODEL_COLORS[model])

ax.set_xlabel('SMAPE (%)  ← 낮을수록 정확', fontsize=10)
ax.set_ylabel('TotalCost (BRL)  ← 낮을수록 효율적', fontsize=10)
ax.set_title('SMAPE × TotalCost 2×2 Matrix\n(◆ = 모델 평균)', fontsize=12)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 5. 레이더 차트 — 다차원 모델 비교

단일 지표가 아닌 6개 축으로 모델을 비교합니다.

| 축 | 설명 | 방향 |
|---|---|---|
| SMAPE | 예측 정확도 | 낮을수록 좋음 |
| TotalCost | 총 운영 비용 | 낮을수록 좋음 |
| HoldingRatio | 비용 중 보관 비율 | 낮을수록 좋음 |
| StockoutRatio | 비용 중 품절 비율 | 낮을수록 좋음 |
| CostVsPerfect | Perfect 대비 초과 비용 배율 | 낮을수록 좋음 |
| WinRate | 카테고리별 1위 비율 | 높을수록 좋음 |

In [ ]:
# ── 6개 지표 계산 ─────────────────────────────────────
metrics = {}
for model in MODELS:
    sub         = summary[summary['model'] == model]
    total_cost  = sub['total_cost'].sum()
    hold_cost   = sub['holding_cost'].sum()
    stock_cost  = sub['stockout_cost'].sum()
    perfect_sum = sub['perfect_cost'].sum()
    win_rate    = (cat_summary['best_cost_model'] == model).mean()

    metrics[model] = {
        'SMAPE'          : sub['smape'].mean(),
        'TotalCost'      : total_cost,
        'HoldingRatio'   : hold_cost  / total_cost * 100,
        'StockoutRatio'  : stock_cost / total_cost * 100,
        'CostVsPerfect'  : total_cost / (perfect_sum + 1e-9),
        'WinRate'        : win_rate * 100,
    }

print("6개 지표 요약:")
print(pd.DataFrame(metrics).T.round(2).to_string())

# ── 레이더 차트 ──────────────────────────────────────
# 각 축을 [0, 1]로 정규화 (낮을수록 좋은 지표는 반전)
AXES      = ['SMAPE', 'TotalCost', 'HoldingRatio', 'StockoutRatio', 'CostVsPerfect', 'WinRate']
INVERT    = ['SMAPE', 'TotalCost', 'HoldingRatio', 'StockoutRatio', 'CostVsPerfect']  # 낮을수록 좋음

raw_vals  = {ax: [metrics[m][ax] for m in MODELS] for ax in AXES}
norm_vals = {}
for ax in AXES:
    vals    = np.array(raw_vals[ax], dtype=float)
    mn, mx  = vals.min(), vals.max()
    normed  = (vals - mn) / (mx - mn + 1e-9)
    norm_vals[ax] = 1 - normed if ax in INVERT else normed

N      = len(AXES)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for model in MODELS:
    i      = MODELS.index(model)
    values = [norm_vals[ax][i] for ax in AXES]
    values += values[:1]
    ax.plot(angles, values, color=MODEL_COLORS[model], linewidth=2, label=model)
    ax.fill(angles, values, color=MODEL_COLORS[model], alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(AXES, fontsize=9)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%','50%','75%','100%'], fontsize=7)
ax.set_title('모델 다차원 비교 레이더 차트\n(외곽일수록 좋음)', pad=20, fontsize=11)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.tight_layout()
plt.show()

## 6. 카테고리별 최적 모델 지도

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 좌: SMAPE 기준 최적 모델 ─────────────────────────
ax = axes[0]
smape_pivot = summary.pivot(index='category', columns='model', values='smape')
best_smape  = smape_pivot.idxmin(axis=1)
colors_s    = [MODEL_COLORS[m] for m in best_smape]

bars = ax.barh(smape_pivot.index,
               smape_pivot.min(axis=1),
               color=colors_s, alpha=0.85)

for bar, (cat, model) in zip(bars, best_smape.items()):
    ax.text(bar.get_width()*0.02, bar.get_y()+bar.get_height()/2,
            model, va='center', fontsize=8, color='white', fontweight='bold')

ax.set_xlabel('최저 SMAPE (%)')
ax.set_title('SMAPE 기준 카테고리별 최적 모델')
legend_handles = [mpatches.Patch(color=MODEL_COLORS[m], label=m) for m in MODELS]
ax.legend(handles=legend_handles, fontsize=8)

# ── 우: TotalCost 기준 최적 모델 ────────────────────
ax = axes[1]
cost_pivot = summary.pivot(index='category', columns='model', values='total_cost')
best_cost  = cost_pivot.idxmin(axis=1)
colors_c   = [MODEL_COLORS[m] for m in best_cost]

bars = ax.barh(cost_pivot.index,
               cost_pivot.min(axis=1),
               color=colors_c, alpha=0.85)

for bar, (cat, model) in zip(bars, best_cost.items()):
    ax.text(bar.get_width()*0.02, bar.get_y()+bar.get_height()/2,
            model, va='center', fontsize=8, color='white', fontweight='bold')

ax.set_xlabel('최저 TotalCost (BRL)')
ax.set_title('TotalCost 기준 카테고리별 최적 모델')
ax.legend(handles=legend_handles, fontsize=8)

plt.suptitle('카테고리별 최적 모델 — SMAPE vs TotalCost 비교', y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

# ── 역전 현황 출력 ────────────────────────────────────
print("\n카테고리별 최적 모델 비교 (SMAPE 1위 vs 비용 1위)")
print("=" * 58)
print(f"{'카테고리':<30} {'SMAPE 1위':<12} {'비용 1위':<12} {'역전'}")
print("-" * 58)
for _, row in cat_summary.iterrows():
    mark = '🔄 역전' if row['reversed'] else '✓  일치'
    print(f"{row['category'][:28]:<30} {row['best_smape_model']:<12} {row['best_cost_model']:<12} {mark}")

n_rev = cat_summary['reversed'].sum()
print(f"\n역전 발생: {n_rev}/{len(CATEGORIES)}개 카테고리 ({n_rev/len(CATEGORIES)*100:.0f}%)")

## 7. 비용 절감 효과 분석

Naive 모델(전일값 발주) 대비 각 모델이 비용을 얼마나 절감했는지 계산합니다.

In [ ]:
saving_rows = []
for cat in CATEGORIES:
    naive_cost = cost_results[cat]['Naive']['total_sum']
    perf_cost  = cost_results[cat]['Perfect']['total_sum']
    for model in MODELS:
        model_cost = cost_results[cat][model]['total_sum']
        saving_rows.append({
            'category'         : cat,
            'model'            : model,
            'model_cost'       : model_cost,
            'naive_cost'       : naive_cost,
            'perfect_cost'     : perf_cost,
            'saving_vs_naive'  : naive_cost  - model_cost,          # 절감액
            'saving_pct'       : (naive_cost - model_cost) / (naive_cost + 1e-9) * 100,  # 절감율
            'gap_to_perfect'   : model_cost  - perf_cost,           # 이론적 개선 여지
            'efficiency'       : perf_cost / (model_cost + 1e-9) * 100,  # Perfect 대비 효율(%)
        })

saving_df = pd.DataFrame(saving_rows)

# ── 전체 합산 절감 효과 ─────────────────────────────
print("Naive 대비 비용 절감 효과 (전체 카테고리 합산)")
print("=" * 60)
for model in MODELS:
    sub        = saving_df[saving_df['model'] == model]
    total_save = sub['saving_vs_naive'].sum()
    avg_pct    = sub['saving_pct'].mean()
    avg_eff    = sub['efficiency'].mean()
    print(f"  {model:<10}: 절감액={total_save:>10,.1f} BRL  절감율={avg_pct:>6.1f}%  Perfect효율={avg_eff:.1f}%")

# ── 시각화 ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 좌: 카테고리별 절감율 비교
ax  = axes[0]
x   = np.arange(len(CATEGORIES))
w   = 0.25
for i, model in enumerate(MODELS):
    sub = saving_df[saving_df['model']==model].set_index('category')
    vals = [sub.loc[c,'saving_pct'] if c in sub.index else 0 for c in CATEGORIES]
    ax.bar(x + i*w, vals, width=w, label=model,
           color=MODEL_COLORS[model], alpha=0.85)

ax.axhline(0, color='black', linewidth=0.7)
ax.set_xticks(x + w)
ax.set_xticklabels([c[:14] for c in CATEGORIES], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Naive 대비 절감율 (%)')
ax.set_title('카테고리별 Naive 대비 비용 절감율')
ax.legend(fontsize=8)

# 우: Perfect 대비 효율 (100% = Perfect와 동일)
ax = axes[1]
for i, model in enumerate(MODELS):
    sub = saving_df[saving_df['model']==model].set_index('category')
    vals = [sub.loc[c,'efficiency'] if c in sub.index else 0 for c in CATEGORIES]
    ax.bar(x + i*w, vals, width=w, label=model,
           color=MODEL_COLORS[model], alpha=0.85)

ax.axhline(100, color='#7F77DD', linewidth=1, linestyle='--', label='Perfect (100%)')
ax.set_xticks(x + w)
ax.set_xticklabels([c[:14] for c in CATEGORIES], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Perfect 대비 효율 (%)')
ax.set_title('카테고리별 Perfect 대비 비용 효율')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 8. 핵심 질문 — 데이터 기반 답변

> **"예측 정확도(SMAPE)가 높을수록 실제 운영 비용도 반드시 감소하는가?"**

In [ ]:
# ── 분석에 필요한 수치 계산 ───────────────────────────
pearson_r,  p_pearson  = pearsonr(summary['smape'],  summary['total_cost'])
spearman_r, p_spearman = spearmanr(summary['smape'], summary['total_cost'])
n_reversed = cat_summary['reversed'].sum()
n_total    = len(cat_summary)

# 모델 전체 평균
model_avg = summary.groupby('model')[['smape','total_cost']].mean()
best_smape_overall = model_avg['smape'].idxmin()
best_cost_overall  = model_avg['total_cost'].idxmin()
rank_match = best_smape_overall == best_cost_overall

# 민감도 (h·s 변화 시 순위 변동)
from scipy.stats import norm as scipy_norm

sensitivity_winners = []
for h_sc in [0.05, 0.10, 0.20]:
    for s_rt in [2.0, 3.0, 5.0]:
        totals = {m: 0 for m in MODELS}
        for cat in CATEGORIES:
            sub = pred_df[pred_df['category']==cat].dropna(
                subset=list(pred_col_map.values())).sort_values('date')
            actual = sub['actual'].values
            avg_f  = freight_by_cat.get(cat, freight_by_cat.mean())
            h_v, s_v = avg_f*h_sc, avg_f*h_sc*s_rt
            rs = pd.Series(actual).rolling(7,min_periods=2).std().fillna(1).values
            for model, col in pred_col_map.items():
                o = order_quantity(sub[col].values, STRATEGY, h_v, s_v, rs)
                totals[model] += simulate_cost(actual, o, h_v, s_v)['total_sum']
        sensitivity_winners.append(min(totals, key=totals.get))

dominant_cost_model = max(set(sensitivity_winners), key=sensitivity_winners.count)
dominant_count      = sensitivity_winners.count(dominant_cost_model)

In [ ]:
# ── 핵심 질문 시각화 대시보드 ────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ① 상관관계 산점도 (좌상)
ax1 = fig.add_subplot(gs[0, 0])
for model in MODELS:
    sub = summary[summary['model']==model]
    ax1.scatter(sub['smape'], sub['total_cost'],
                color=MODEL_COLORS[model], s=55, alpha=0.85, label=model, zorder=3)
z  = np.polyfit(summary['smape'], summary['total_cost'], 1)
xr = np.linspace(summary['smape'].min(), summary['smape'].max(), 100)
ax1.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=0.8, alpha=0.5)
ax1.set_xlabel('SMAPE (%)')
ax1.set_ylabel('TotalCost (BRL)')
ax1.set_title(f'① SMAPE vs TotalCost\nPearson r={pearson_r:.3f}  p={p_pearson:.3f}')
ax1.legend(fontsize=7)

# ② 모델 전체 평균 비교 (중상)
ax2 = fig.add_subplot(gs[0, 1])
x_pos = np.arange(len(MODELS))
colors_bar = [MODEL_COLORS[m] for m in MODELS]

smape_vals = [model_avg.loc[m,'smape']      for m in MODELS]
cost_vals  = [model_avg.loc[m,'total_cost'] for m in MODELS]

ax2b = ax2.twinx()
ax2.bar(x_pos - 0.2, smape_vals, 0.35, color=colors_bar, alpha=0.5, label='SMAPE (좌)')
ax2b.bar(x_pos + 0.2, cost_vals, 0.35, color=colors_bar, alpha=0.9, label='TotalCost (우)')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(MODELS)
ax2.set_ylabel('SMAPE (%)', fontsize=8)
ax2b.set_ylabel('TotalCost (BRL)', fontsize=8)
ax2.set_title('② 모델별 전체 평균\nSMAPE vs TotalCost')

# 1위 모델 강조
smape_best_idx = smape_vals.index(min(smape_vals))
cost_best_idx  = cost_vals.index(min(cost_vals))
ax2.get_children()[smape_best_idx].set_edgecolor('black')
ax2.get_children()[smape_best_idx].set_linewidth(2)

# ③ 역전 카테고리 (우상)
ax3 = fig.add_subplot(gs[0, 2])
rev_colors = ['#D85A30' if r else '#1D9E75' for r in cat_summary['reversed']]
ax3.barh(range(n_total), [1]*n_total, color=rev_colors, alpha=0.75)
ax3.set_yticks(range(n_total))
ax3.set_yticklabels(
    [f"{r['category'][:16]}  {r['best_smape_model']}→{r['best_cost_model']}"
     for _, r in cat_summary.iterrows()], fontsize=7)
ax3.set_xticks([])
ax3.set_title(f'③ 순위 역전 현황\n역전={n_reversed}/{n_total} ({n_reversed/n_total*100:.0f}%)')
ax3.legend(handles=[
    mpatches.Patch(color='#1D9E75', label='일치'),
    mpatches.Patch(color='#D85A30', label='역전'),
], fontsize=7)

# ④ 누적 비용 (좌하)
ax4 = fig.add_subplot(gs[1, 0])
daily_agg = cost_daily_df.groupby(['date','model'])['total'].sum().reset_index()
for model in MODELS:
    sub = daily_agg[daily_agg['model']==model].sort_values('date')
    ax4.plot(sub['date'], sub['total'].cumsum(),
             color=MODEL_COLORS[model], linewidth=1.5, label=model)
ax4.set_ylabel('누적 TotalCost (BRL)')
ax4.set_title('④ 누적 비용 추이')
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax4.tick_params(axis='x', rotation=40, labelsize=7)
ax4.legend(fontsize=7)

# ⑤ 민감도 (중하)
ax5 = fig.add_subplot(gs[1, 1])
winner_counts = {m: sensitivity_winners.count(m) for m in MODELS}
ax5.bar(MODELS,
        [winner_counts.get(m,0) for m in MODELS],
        color=colors_bar, alpha=0.85)
ax5.set_ylabel('h·s 시나리오 1위 횟수')
ax5.set_title(f'⑤ h·s 민감도 분석\n(총 {len(sensitivity_winners)}개 시나리오)')
for i, m in enumerate(MODELS):
    ax5.text(i, winner_counts.get(m,0)+0.05,
             str(winner_counts.get(m,0)), ha='center', fontsize=10, fontweight='bold')

# ⑥ 텍스트 결론 (우하)
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')

if abs(pearson_r) < 0.3:
    corr_str = f"약한 상관관계 (r={pearson_r:.2f})"
    corr_ans = "SMAPE↑이라도 비용↓이 가능함"
elif abs(pearson_r) < 0.6:
    corr_str = f"중간 상관관계 (r={pearson_r:.2f})"
    corr_ans = "대체로 연관되나 예외 존재"
else:
    corr_str = f"강한 상관관계 (r={pearson_r:.2f})"
    corr_ans = "SMAPE와 비용이 함께 움직임"

conclusion_text = (
    f"  핵심 질문 답변\n"
    f"{'─'*34}\n"
    f"  Q. SMAPE↓ → 비용↓ 이 성립하는가?\n\n"
    f"  상관관계: {corr_str}\n"
    f"  → {corr_ans}\n\n"
    f"  순위 역전: {n_reversed}/{n_total}개 카테고리\n"
    f"  → 반드시 성립하지는 않음\n\n"
    f"  SMAPE 전체 1위: {best_smape_overall}\n"
    f"  비용   전체 1위: {best_cost_overall}\n"
    f"  {'✓ 두 지표 1위 일치' if rank_match else '✗ 두 지표 1위 불일치 — 역전 발생'}\n\n"
    f"  민감도 강건 모델: {dominant_cost_model}\n"
    f"  ({dominant_count}/{len(sensitivity_winners)} 시나리오에서 비용 1위)\n"
    f"{'─'*34}\n"
    f"  ※ 최종 권장 모델:\n"
    f"    비용 최소화 최우선 → {best_cost_overall}\n"
    f"    예측 정확도 최우선 → {best_smape_overall}\n"
    f"    파라미터 변화 강건 → {dominant_cost_model}"
)

ax6.text(0.03, 0.97, conclusion_text,
         transform=ax6.transAxes,
         fontsize=8.5, verticalalignment='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.6',
                   facecolor='#F7F6F2', edgecolor='#CCCCCC', linewidth=1))

fig.suptitle('핵심 질문 답변 대시보드\n"예측 정확도(SMAPE)↑ → 운영 비용↓ 이 반드시 성립하는가?"',
             fontsize=13, y=1.01)
plt.savefig('../data/final_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print("대시보드 저장 완료: ../data/final_dashboard.png")

## 9. 프로젝트 최종 요약

In [ ]:
print("=" * 65)
print(" 프로젝트 최종 요약")
print("=" * 65)

print("\n[ 데이터 ]")
print(f"  분석 기간  : {ANALYSIS_START} ~ {ANALYSIS_END}")
print(f"  예측 기간  : {TEST_START} ~ {ANALYSIS_END}")
print(f"  대상 카테고리 : {len(CATEGORIES)}개")

print("\n[ 예측 성능 (SMAPE, 낮을수록 좋음) ]")
for model in MODELS:
    avg = summary[summary['model']==model]['smape'].mean()
    print(f"  {model:<10}: {avg:.2f}%")

print("\n[ 운영 비용 (TotalCost, 낮을수록 좋음) ]")
for model in MODELS + ['Perfect', 'Naive']:
    total = sum(cost_results[c][model]['total_sum'] for c in CATEGORIES)
    print(f"  {model:<10}: {total:>12,.1f} BRL")

print("\n[ 핵심 질문 답변 ]")
print(f"  SMAPE ↔ TotalCost 상관계수 : Pearson r={pearson_r:.3f}, Spearman r={spearman_r:.3f}")
print(f"  순위 역전 발생 카테고리     : {n_reversed}/{n_total}개")
print(f"  SMAPE 전체 1위 모델         : {best_smape_overall}")
print(f"  비용   전체 1위 모델         : {best_cost_overall}")
print(f"  민감도 강건 모델             : {dominant_cost_model}")

if not rank_match:
    print()
    print("  ★ 결론: SMAPE 1위 모델과 비용 1위 모델이 일치하지 않습니다.")
    print("          예측 정확도 향상이 운영 비용 절감을 보장하지 않습니다.")
    print("          비용 최소화를 목표로 한다면 모델 선택 기준을")
    print("          SMAPE가 아닌 TotalCost 직접 최적화로 전환해야 합니다.")
else:
    print()
    print("  ★ 결론: SMAPE 1위 모델이 비용 1위 모델과 일치합니다.")
    print("          단, 일부 카테고리에서는 역전이 발생하므로")
    print("          카테고리별 모델 선택이 전체 단일 모델보다 유리합니다.")

print()
print("=" * 65)
print(" 노트북 실행 순서 완료")
print("  01_EDA.ipynb")
print("  02_feature_engineering.ipynb")
print("  03_modeling.ipynb")
print("  04_smape_evaluation.ipynb")
print("  05_cost_simulation.ipynb")
print("  06_final_comparison.ipynb  ← 현재")
print("=" * 65)